**name - kaveri**
**srn - pes1ug23cs293**

##to identify marine debris (plastic) from multispectral satellite imagery using deep learning.


In [ ]:
!wget -O MARIDA.zip https://zenodo.org/record/5151941/files/MARIDA.zip?download=1

: 

In [ ]:
import zipfile
import os

# 2. Unzip the dataset
print("Unzipping dataset... this may take a minute.")

# Check if file exists before unzipping
if os.path.exists("MARIDA.zip"):
    with zipfile.ZipFile("MARIDA.zip", 'r') as zip_ref:
        zip_ref.extractall("./MARIDA_Dataset")
    print("Dataset downloaded and extracted to ./MARIDA_Dataset")
else:
    print("Error: MARIDA.zip not found. Please ensure the download step completed successfully.")

# 3. Verify structure
base_dir = "./MARIDA_Dataset"
if os.path.exists(base_dir):
    # Check what is inside to ensure we have the 'patches' folder
    print(f"Found contents: {os.listdir(base_dir)}")

In [ ]:
import os

patches_dir = "./MARIDA_Dataset/patches"
print("Number of patch files:", len(os.listdir(patches_dir)))
print("Sample files:", os.listdir(patches_dir)[:5])


In [ ]:
import os

patch_name = os.listdir(patches_dir)[0]
patch_path = os.path.join(patches_dir, patch_name)

print("Patch folder:", patch_name)
print("Contents:", os.listdir(patch_path))


In [ ]:
#TASK 3
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=True)


In [ ]:
old_conv = model.conv1
old_weights = old_conv.weight.data  # shape: [64, 3, 7, 7]


In [ ]:
new_conv = nn.Conv2d(
    in_channels=11,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)


In [ ]:
with torch.no_grad():
    # Copy RGB weights directly
    new_conv.weight[:, :3, :, :] = old_weights

    # Average RGB weights for remaining bands
    avg_weight = old_weights.mean(dim=1, keepdim=True)
    for i in range(3, 11):
        new_conv.weight[:, i:i+1, :, :] = avg_weight


In [ ]:
model.conv1 = new_conv


In [ ]:
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)


In [ ]:
#TASK 4
import os
import re
import rasterio
import numpy as np
import torch
from torch.utils.data import Dataset

# --- Paths ---
PATCHES_DIR = "/content/MARIDA_Dataset/patches"

# ---- Helpers ----
def sort_band_files(files):
    valid = [f for f in files if re.search(r'_(\d+)_conf', f)]
    return sorted(valid, key=lambda x: int(re.search(r'_(\d+)_conf', x).group(1)))

def min_max_normalize(img):
    min_val = img.min(axis=(1,2), keepdims=True)
    max_val = img.max(axis=(1,2), keepdims=True)
    return (img - min_val) / (max_val - min_val + 1e-6)

# ---- Dataset ----
class MARIDADataset(Dataset):
    def __init__(self, patches_dir):
        self.patches_dir = patches_dir
        self.patch_names = [
            d for d in os.listdir(patches_dir)
            if os.path.isdir(os.path.join(patches_dir, d))
        ]

    def __len__(self):
        return len(self.patch_names)

    def __getitem__(self, idx):
        patch_name = self.patch_names[idx]
        patch_path = os.path.join(self.patches_dir, patch_name)

        #  FORCE EXACTLY 11 BANDS
        band_files = sort_band_files(os.listdir(patch_path))[:11]

        bands = []
        for bf in band_files:
            with rasterio.open(os.path.join(patch_path, bf)) as src:
                bands.append(src.read(1))

        img = np.stack(bands, axis=0)

        #  PAD IF LESS THAN 11
        if img.shape[0] < 11:
            pad = np.zeros((11 - img.shape[0], img.shape[1], img.shape[2]))
            img = np.concatenate([img, pad], axis=0)

        img = min_max_normalize(img)
        img = torch.tensor(img, dtype=torch.float32)

        with rasterio.open(os.path.join(patch_path, band_files[0])) as src:
            label = torch.tensor(int(src.tags().get("label", 0)))

        return img, label

# ---- TEST ----
train_dataset = MARIDADataset(PATCHES_DIR)
print("Usable patches:", len(train_dataset))

img, label = train_dataset[0]
print("Image shape:", img.shape)
print("Label:", label)

In [ ]:
train_dataset = MARIDADataset(PATCHES_DIR)
print("Usable patches:", len(train_dataset))

img, label = train_dataset[0]
print("Image shape:", img.shape)
print("Label:", label)


In [ ]:

# TASK 5: TRAINING & RESULTS


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- DataLoader ----
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

# ---- Load ResNet18 (11-channel model already prepared earlier) ----
model = models.resnet18(pretrained=True)

# Modify first conv layer to accept 11 channels
old_conv = model.conv1
new_conv = nn.Conv2d(
    in_channels=11,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

with torch.no_grad():
    new_conv.weight[:, :3, :, :] = old_conv.weight
    avg_weight = old_conv.weight.mean(dim=1, keepdim=True)
    for i in range(3, 11):
        new_conv.weight[:, i:i+1, :, :] = avg_weight

model.conv1 = new_conv

# Replace classification head (binary classification)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

# ---- Loss & Optimizer ----
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ---- Training Loop ----
epochs = 10

print("Starting training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}, Accuracy: {accuracy:.2f}%")

print("Training completed.")


In [ ]:

# TASK 6: GRAD-CAM (MODEL EXPLAINABILITY)


import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# ---- Set model to eval mode ----
model.eval()

# ---- Grad-CAM class ----
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, x, class_idx):
        output = self.model(x)
        self.model.zero_grad()
        output[0, class_idx].backward()

        # Global Average Pooling on gradients
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = F.relu(cam)

        # Normalize CAM
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-6)
        return cam

# ---- Initialize Grad-CAM on last conv layer ----
grad_cam = GradCAM(model, model.layer4)

# ---- Get one sample ----
img, label = train_dataset[0]
input_img = img.unsqueeze(0).to(device)

# ---- Generate CAM ----
cam = grad_cam.generate(input_img, label.item())
cam = cam.cpu().detach().numpy()[0]

# ---- Convert multispectral image to RGB for visualization ----
# (use first 3 bands as pseudo-RGB)
rgb = img[:3].permute(1, 2, 0).numpy()
rgb = (rgb - rgb.min()) / (rgb.max() + 1e-6)

# ---- Plot ----
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.title("Input Image (Pseudo-RGB)")
plt.imshow(rgb)
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Grad-CAM Heatmap")
plt.imshow(rgb)
plt.imshow(cam, cmap="jet", alpha=0.5)
plt.axis("off")

plt.show()


Grad-CAM was used to visualize the spatial regions influencing the model’s decision. It highlights areas contributing most to marine debris detection, improving model interpretability.